Prompt Chaining with the langgraph. ( using the multiple prompt )

For this workflow . We will give a topic and generate the blog: 
For this: We will ask the llm for the outline of the blog with the first prompt . The we will generate the blog using the detailed outline  generated from the llm 

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

/home/foolmann/miniconda3/envs/genaienv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [20]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    rating: int
    

In [6]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title

    title = state['title']

    # call thellm gen outline 

    prompt = f'Generate a detailed but not long outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content[0]['text']
    # update state 
    state['outline'] = outline

    return state 

In [7]:
def create_blog(state: BlogState) -> BlogState:

    # fetch title

    title = state['title']

    # call thellm gen blog
    outline= state['outline']

    prompt = f'Generate a detailed blog on the topic - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content[0]['text']
   
    # update state 
    state['content'] = content

    return state 

In [14]:
def evaluate_blog(state: BlogState) -> BlogState:

    # fetch title

    title = state['title']

    # call thellm gen blog
    blog= state['content']

    prompt = f'Evaluate the  blog on the topic - {title} and content \n {blog} and give rating on 5 stars . ( Note the rating must be int between 1-5)'

    rating = model.invoke(prompt).content[0]['text']
   
    # update state 
    state['rating'] = rating

    return state 

In [16]:
graph = StateGraph(BlogState)

# nodes 

graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('evaluate_blog',evaluate_blog)


# edges 

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','evaluate_blog')
graph.add_edge('evaluate_blog',END)

# compile 

workflow = graph.compile()

In [17]:
initial_state = {'title':'Nims Purja BroadPeak Incident'}
final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Nims Purja BroadPeak Incident', 'outline': 'This outline provides a structured, balanced, and engaging approach to discussing the Nims Purja Broad Peak incident, focusing on the intersection of high-altitude mountaineering, ethics, and social media influence.\n\n---\n\n### **Blog Title Ideas:**\n*   *The Shadow of the Summit: Examining the Nims Purja Broad Peak Controversy*\n*   *Ethics at 8,000 Meters: What Really Happened on Broad Peak?*\n*   *Broad Peak and the Burden of Heroism: A Closer Look at the Nimsdai Incident*\n\n---\n\n### **Outline**\n\n#### **I. Introduction**\n*   **The Hook:** Briefly describe the allure and danger of the "Death Zone" on Broad Peak.\n*   **The Context:** Introduce Nimsdai Purja—the record-breaking mountaineer—and the 2023 controversy involving a stranded climber.\n*   **Thesis Statement:** Acknowledge that the incident sparked a heated debate regarding rescue responsibilities, the "hero" narrative in mountaineering, and the ethics of commerci

In [18]:
print(final_state['content'])

# The Shadow of the Summit: Examining the Nims Purja Broad Peak Controversy

### I. Introduction
Broad Peak, the 12th highest mountain in the world, is a place where the thin air of the "Death Zone"—anything above 8,000 meters—turns every step into an act of supreme physical endurance. It is a place of breathtaking beauty and chilling indifference. In 2023, the mountain became the center of a firestorm involving one of the most recognizable figures in modern alpinism: Nimsdai "Nims" Purja.

Nims, famous for his record-breaking *14 Peaks* mission, is a titan of the sport. However, during his 2023 expedition to Broad Peak, an incident involving a distressed climber brought his professional and personal ethics under intense public scrutiny. This controversy serves as a stark lens through which to view the current state of high-altitude mountaineering, raising difficult questions about rescue responsibilities, the "hero" narrative, and the ethics of commercial operations in the world’s mos

In [19]:
print(final_state['rating'])


This is a well-structured, thought-provoking, and balanced piece of journalism. It navigates a highly contentious subject without succumbing to the "clickbait" style often found in sports commentary.

### Evaluation

**Strengths:**
*   **Tone and Objectivity:** The greatest strength of this blog is its neutrality. It acknowledges Nimsdai Purja’s status as a "titan of the sport" while respecting the gravity of the accusations. By framing the incident as a "polarizing gray area," you avoid taking a definitive side, which allows the reader to engage with the moral complexities rather than just picking a team.
*   **Contextual Framing:** You successfully link a specific incident to a broader systemic issue—the commercialization of high-altitude mountaineering. This elevates the post from a piece of "gossip" about a celebrity climber to a serious analysis of the sport's ethics.
*   **Clarity of Argument:** Section III ("The Core Debate") is excellent. It clearly defines the tension between 